# 📊 SOS — Strength of Schedule
### Enriquecimento do `rodada_atual_v3.csv` com métricas de força da tabela

---
**O que este notebook faz:**
1. Lê `rodada_atual_v3.csv` e `base_historica_v3.csv`
2. Calcula o SOS (força dos adversários enfrentados) para cada time via ELO
3. Calcula Z-Scores brutos e ajustados por SOS (Z_adj)
4. Gera `rodada_atual_v3.csv` atualizado com 14 novas colunas

---
**Equações:**
```
SOS_raw  = Σ(ELO_adversário_i × decay^i) / Σ(decay^i)   [últimos N jogos]
SOS_norm = SOS_raw / ELO_médio_liga                       [1.0 = média da liga]
Z_adj    = Z_bruto / SOS_norm                             [tabela fácil amplifica alerta]
```

**Interpretação SOS_norm:**
- `> 1.05` → Tabela difícil — desempenho mais legítimo, Z menos preocupante
- `0.95–1.05` → Tabela normal
- `< 0.95` → Tabela fácil — desempenho inflado, Z_adj sobe

In [49]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 0 — Imports e Constantes
# ═══════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from IPython.display import display

# ── Caminhos ──────────────────────────────────────────────
PATH_RODADA   = 'rodada_atual_v3.csv'      # arquivo de entrada (rodada atual)
PATH_BASE     = 'base_historica_v3.csv'    # base histórica com ELO por jogo
PATH_OUTPUT   = 'rodada_atual_sos_v3.csv'      # sobrescreve com novas colunas

# ── Parâmetros SOS ────────────────────────────────────────
N_JOGOS      = 10      # últimos N jogos para calcular SOS
DECAY        = 0.85    # fator de decay — jogo mais recente = peso 1.0
                        # jogo anterior = 0.85, anterior a esse = 0.72, etc.

# ── Parâmetros RTM (Regressão à Média) calibrados na base histórica ──
# μ e σ calculados sobre todos os jogos com N10 preenchido (n≈5.000)
RTM_PARAMS = {
    'gols_home': {'mu': 1.5336, 'sigma': 0.6711, 'col': 'gols_marc_home'},
    'gols_away': {'mu': 1.2656, 'sigma': 0.5790, 'col': 'gols_marc_away'},
    'xg_home':   {'mu': 1.5592, 'sigma': 0.3601, 'col': 'xg_home_n10'},
    'xg_away':   {'mu': 1.3208, 'sigma': 0.3242, 'col': 'xg_away_n10'},
    'vit_home':  {'mu': 0.5002, 'sigma': 0.2456, 'col': 'pct_vit_home'},
    'vit_away':  {'mu': 0.3916, 'sigma': 0.2172, 'col': 'pct_vit_away'},
}

print('✅ Constantes configuradas')
print(f'   N_JOGOS={N_JOGOS} | DECAY={DECAY}')
print(f'   PATH_OUTPUT: {PATH_OUTPUT}')

✅ Constantes configuradas
   N_JOGOS=10 | DECAY=0.85
   PATH_OUTPUT: rodada_atual_sos_v3.csv


In [50]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 1 — Carregar CSVs
# ═══════════════════════════════════════════════════════════
ra = pd.read_csv(PATH_RODADA)
bh = pd.read_csv(PATH_BASE)

print(f'rodada_atual : {ra.shape[0]} jogos | {ra.shape[1]} colunas')
print(f'base_historica: {bh.shape[0]} jogos | {bh.shape[1]} colunas')
print(f'Ligas rodada_atual : {ra["league"].nunique()}')
print(f'Ligas base_historica: {bh["league"].nunique()}')
print()

# Verificar se já tem colunas SOS (evitar recalcular desnecessariamente)
colunas_sos = [c for c in ra.columns if 'sos' in c.lower() or c.startswith('z_')]
if colunas_sos:
    print(f'⚠️  Colunas SOS/Z já existem: {colunas_sos}')
    print('   Serão recalculadas e sobrescritas.')
else:
    print('ℹ️  Nenhuma coluna SOS/Z encontrada — calculando do zero.')

# Cobertura de IDs
ids_ra = set(ra['home_id'].tolist() + ra['away_id'].tolist())
ids_bh = set(bh['home_id'].dropna().tolist()) | set(bh['away_id'].dropna().tolist())
cobertura = len(ids_ra & ids_bh) / len(ids_ra) * 100
print(f'Cobertura de times por ID: {cobertura:.0f}%')

display(ra[['league','home_team','away_team','elo_home','elo_away']].head(5))

rodada_atual : 154 jogos | 62 colunas
base_historica: 5612 jogos | 78 colunas
Ligas rodada_atual : 25
Ligas base_historica: 28

ℹ️  Nenhuma coluna SOS/Z encontrada — calculando do zero.
Cobertura de times por ID: 100%


,league,home_team,away_team,elo_home,elo_away
0,Alemanha_1 Bundesliga,Bayern München,Union Berlin,1687.972611,1469.677685
1,Alemanha_1 Bundesliga,Mainz 05,Eintracht Frankfurt,1487.626705,1509.830258
2,Alemanha_1 Bundesliga,Augsburg,Stuttgart,1472.283854,1587.061859
3,Alemanha_1 Bundesliga,St. Pauli,Freiburg,1431.297981,1487.521427
4,Alemanha_1 Bundesliga,Borussia Dortmund,Hamburger SV,1641.547269,1475.946694


In [51]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 2 — Calcular ELO médio por liga (base de normalização)
# ═══════════════════════════════════════════════════════════

# Nota: no sistema ELO zero-sum, a média de uma liga tende a 1500
# Calculamos mesmo assim para capturar ligeiros desvios reais
elo_liga = {}
for liga in bh['league'].unique():
    sub  = bh[bh['league'] == liga]
    elos = pd.concat([sub['elo_home'], sub['elo_away']]).dropna()
    # Excluir ELO padrão inicial (1500.0 exato) que distorce a média
    elos_reais = elos[elos != 1500.0]
    elo_liga[liga] = float(elos_reais.mean()) if len(elos_reais) > 10 else 1500.0

print(f'ELO médio calculado para {len(elo_liga)} ligas:')
for liga, elo in sorted(elo_liga.items()):
    print(f'  {liga:<40} {elo:.1f}')

ELO médio calculado para 28 ligas:
  Alemanha_1 Bundesliga                    1500.0
  Alemanha_2 2. Bundesliga                 1500.3
  Alemanha_3 3. Liga                       1498.5
  Argentina_1 Liga Profesional             1499.8
  Australia_1 A-League                     1500.2
  Belgica_1 Pro League                     1500.2
  Brasileiro_A Serie A                     1499.9
  Bulgaria_1 First League                  1499.8
  Chile_1 Primera Division                 1500.0
  Colombia_1 Liga BetPlay                  1500.7
  Croacia_1 HNL                            1500.1
  Escocia_1 Premiership                    1500.1
  Espanha_1 La Liga                        1498.9
  Espanha_2 Segunda Division               1500.2
  Franca_1 Ligue 1                         1499.7
  Franca_2 Ligue 2                         1499.1
  Grecia_1 Super League                    1500.1
  Holanda_1 Eredivisie                     1500.0
  Holanda_2 Eerste Divisie                 1500.5
  Hungria_1 NB 

In [52]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 3 — Funções de cálculo SOS e Z-Score
# ═══════════════════════════════════════════════════════════

def calc_sos(team_id, liga, bh_df, elo_liga_map, n=N_JOGOS, decay=DECAY):
    """
    Calcula SOS_raw e SOS_norm para um time.
    
    Estratégia:
    - Pega os últimos N jogos do time (casa + fora)
    - Usa o ELO do adversário em cada jogo
    - Aplica decay exponencial (jogo mais recente = peso maior)
    - Normaliza pelo ELO médio da liga
    
    Retorna: (sos_raw, sos_norm, n_jogos_usados)
    """
    # Jogos como mandante → adversário = time de fora (elo_away)
    j_casa = bh_df[bh_df['home_id'] == team_id][['date_unix', 'elo_away']].copy()
    j_casa.columns = ['date_unix', 'opp_elo']
    
    # Jogos como visitante → adversário = time de casa (elo_home)
    j_fora = bh_df[bh_df['away_id'] == team_id][['date_unix', 'elo_home']].copy()
    j_fora.columns = ['date_unix', 'opp_elo']
    
    # Combinar, filtrar ELO válido e ordenar por mais recente
    todos = (
        pd.concat([j_casa, j_fora])
        .dropna(subset=['opp_elo'])
        .query('opp_elo > 1000')      # remover defaults inválidos
        .sort_values('date_unix', ascending=False)
        .head(n)
        .reset_index(drop=True)
    )
    
    if len(todos) == 0:
        return None, None, 0
    
    # Pesos decay: índice 0 (mais recente) = 1.0, índice 1 = 0.85, etc.
    pesos   = np.array([decay**i for i in range(len(todos))])
    sos_raw = float(np.average(todos['opp_elo'], weights=pesos))
    elo_med = elo_liga_map.get(liga, 1500.0)
    sos_norm = sos_raw / elo_med if elo_med > 0 else 1.0
    
    return round(sos_raw, 2), round(sos_norm, 4), len(todos)


def z_score(val, mu, sigma):
    """Z = (x - μ) / σ"""
    return round((val - mu) / sigma, 4) if sigma > 0 else 0.0


def z_ajustado(z_bruto, sos_norm):
    """
    Z_adj = Z_bruto / SOS_norm
    
    Lógica:
    - SOS_norm < 1 (tabela fácil)  → Z_adj > Z_bruto  → alerta sobe
    - SOS_norm = 1 (tabela normal) → Z_adj = Z_bruto   → sem alteração
    - SOS_norm > 1 (tabela difícil)→ Z_adj < Z_bruto   → alerta cai (merecer)
    """
    sos = sos_norm if sos_norm and sos_norm > 0 else 1.0
    return round(z_bruto / sos, 4)


# Teste rápido com Bayern München
id_teste  = ra.iloc[0]['home_id']
nom_teste = ra.iloc[0]['home_team']
liga_teste= ra.iloc[0]['league']
raw, norm, n = calc_sos(id_teste, liga_teste, bh, elo_liga)
print(f'Teste — {nom_teste}:')
print(f'  SOS_raw={raw} | SOS_norm={norm} | n_jogos={n}')
print(f'  ELO médio liga: {elo_liga[liga_teste]:.1f}')
print()
print('✅ Funções definidas')

Teste — Bayern München:
  SOS_raw=1518.21 | SOS_norm=1.0122 | n_jogos=10
  ELO médio liga: 1500.0

✅ Funções definidas


In [53]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 4 — Calcular SOS para todos os times da rodada
# ═══════════════════════════════════════════════════════════

print(f'Calculando SOS para {len(ra)} jogos ({len(ra)*2} times)...')

sos_cols = {
    'sos_home_raw':  [], 'sos_home_norm': [], 'sos_home_n': [],
    'sos_away_raw':  [], 'sos_away_norm': [], 'sos_away_n': [],
}

for _, row in ra.iterrows():
    rh, nh, ch = calc_sos(row['home_id'], row['league'], bh, elo_liga)
    ra_r, na, ca = calc_sos(row['away_id'], row['league'], bh, elo_liga)
    sos_cols['sos_home_raw'].append(rh)
    sos_cols['sos_home_norm'].append(nh)
    sos_cols['sos_home_n'].append(ch)
    sos_cols['sos_away_raw'].append(ra_r)
    sos_cols['sos_away_norm'].append(na)
    sos_cols['sos_away_n'].append(ca)

# Remover colunas SOS antigas (se existirem) antes de adicionar
for col in sos_cols:
    if col in ra.columns:
        ra.drop(columns=[col], inplace=True)
    ra[col] = sos_cols[col]

# Resumo
nulos_h = ra['sos_home_norm'].isna().sum()
nulos_a = ra['sos_away_norm'].isna().sum()
print(f'Nulos SOS home: {nulos_h} | away: {nulos_a}')
print(f'SOS_norm home: min={ra["sos_home_norm"].min():.4f} | max={ra["sos_home_norm"].max():.4f} | μ={ra["sos_home_norm"].mean():.4f}')
print(f'SOS_norm away: min={ra["sos_away_norm"].min():.4f} | max={ra["sos_away_norm"].max():.4f} | μ={ra["sos_away_norm"].mean():.4f}')
print()

display(ra[['league','home_team','away_team',
            'sos_home_norm','sos_home_n',
            'sos_away_norm','sos_away_n']].head(10))

Calculando SOS para 154 jogos (308 times)...
Nulos SOS home: 0 | away: 0
SOS_norm home: min=0.9756 | max=1.0332 | μ=1.0002
SOS_norm away: min=0.9694 | max=1.0262 | μ=1.0001



,league,home_team,away_team,sos_home_norm,sos_home_n,sos_away_norm,sos_away_n
0,Alemanha_1 Bundesliga,Bayern München,Union Berlin,1.0122,10,0.9988,10
1,Alemanha_1 Bundesliga,Mainz 05,Eintracht Frankfurt,1.0075,10,0.9929,10
2,Alemanha_1 Bundesliga,Augsburg,Stuttgart,1.0046,10,0.9783,10
3,Alemanha_1 Bundesliga,St. Pauli,Freiburg,1.0083,10,0.9986,10
4,Alemanha_1 Bundesliga,Borussia Dortmund,Hamburger SV,0.9936,10,0.9792,10
5,Alemanha_1 Bundesliga,Heidenheim,Bayer Leverkusen,1.0057,10,1.0121,10
6,Alemanha_1 Bundesliga,Köln,Borussia M'gladbach,1.0241,10,1.0147,10
7,Alemanha_1 Bundesliga,Wolfsburg,Werder Bremen,1.0101,10,0.9888,10
8,Alemanha_2 2. Bundesliga,Preußen Münster,Magdeburg,0.9981,10,1.0159,10
9,Alemanha_2 2. Bundesliga,Fortuna Düsseldorf,Hertha BSC,0.9985,10,1.0159,10


In [54]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 5 — Calcular Z-Scores e Z ajustados por SOS
# ═══════════════════════════════════════════════════════════

# Z-Scores brutos (sem SOS)
for key, p in RTM_PARAMS.items():
    col_z    = f'z_{key}'
    col_zadj = f'z_{key}_adj'
    
    if p['col'] not in ra.columns:
        print(f'⚠️  Coluna {p["col"]} não encontrada — pulando {key}')
        continue
    
    # Remover se já existir
    for c in [col_z, col_zadj]:
        if c in ra.columns:
            ra.drop(columns=[c], inplace=True)
    
    # Z bruto
    ra[col_z] = ra[p['col']].apply(
        lambda x: z_score(float(x) if pd.notna(x) else p['mu'], p['mu'], p['sigma'])
    )
    
    # Z ajustado por SOS
    # home metrics → ajuste por sos_home_norm
    # away metrics → ajuste por sos_away_norm
    sos_col = 'sos_home_norm' if 'home' in key else 'sos_away_norm'
    ra[col_zadj] = ra.apply(
        lambda r: z_ajustado(r[col_z], r[sos_col]), axis=1
    )

z_cols = [c for c in ra.columns if c.startswith('z_')]
print(f'Colunas Z calculadas: {z_cols}')
print()
print('Amostra comparativa Z_bruto vs Z_adj:')
display(ra[['home_team','away_team',
            'z_gols_home','sos_home_norm','z_gols_home_adj',
            'z_gols_away','sos_away_norm','z_gols_away_adj']].head(10))

Colunas Z calculadas: ['z_gols_home', 'z_gols_home_adj', 'z_gols_away', 'z_gols_away_adj', 'z_xg_home', 'z_xg_home_adj', 'z_xg_away', 'z_xg_away_adj', 'z_vit_home', 'z_vit_home_adj', 'z_vit_away', 'z_vit_away_adj']

Amostra comparativa Z_bruto vs Z_adj:


,home_team,away_team,z_gols_home,sos_home_norm,z_gols_home_adj,z_gols_away,sos_away_norm,z_gols_away_adj
0,Bayern München,Union Berlin,2.1851,1.0122,2.1588,-0.6314,0.9988,-0.6322
1,Mainz 05,Eintracht Frankfurt,0.0989,1.0075,0.0982,0.5775,0.9929,0.5816
2,Augsburg,Stuttgart,-0.1991,1.0046,-0.1982,1.6138,0.9783,1.6496
3,St. Pauli,Freiburg,-0.9441,1.0083,-0.9363,-0.4587,0.9986,-0.4593
4,Borussia Dortmund,Hamburger SV,1.5890,0.9936,1.5992,-0.1133,0.9792,-0.1157
5,Heidenheim,Bayer Leverkusen,-0.9441,1.0057,-0.9387,0.4048,1.0121,0.4000
6,Köln,Borussia M'gladbach,-0.6461,1.0241,-0.6309,-0.8041,1.0147,-0.7925
7,Wolfsburg,Werder Bremen,-0.6461,1.0101,-0.6396,-0.2860,0.9888,-0.2892
8,Preußen Münster,Magdeburg,-0.9441,0.9981,-0.9459,0.9230,1.0159,0.9086
9,Fortuna Düsseldorf,Hertha BSC,-0.6461,0.9985,-0.6471,0.7503,1.0159,0.7386


In [55]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 6 — Ranking SOS e análise de alertas
# ═══════════════════════════════════════════════════════════

def classif_sos(sos_norm):
    if pd.isna(sos_norm): return 'N/A'
    if sos_norm >= 1.03:  return '🔴 Tabela difícil'
    if sos_norm >= 1.01:  return '🟡 Levemente difícil'
    if sos_norm >= 0.99:  return '⚪ Normal'
    if sos_norm >= 0.97:  return '🟡 Levemente fácil'
    return '🟢 Tabela fácil'

ra['sos_home_class'] = ra['sos_home_norm'].apply(classif_sos)
ra['sos_away_class'] = ra['sos_away_norm'].apply(classif_sos)

# Times com maior distorção (tabela fácil + Z_gols alto = alerta máximo)
print('=== Top 10 — Maior Z_gols_home ajustado (mais preocupantes para Over) ===')
top = ra.nlargest(10, 'z_gols_home_adj')[[
    'league','home_team','away_team',
    'gols_marc_home','z_gols_home','sos_home_norm','z_gols_home_adj','sos_home_class'
]]
display(top)

print()
print('=== Times com tabela fácil (SOS_norm < 0.97) — números inflados ===')
facil = ra[ra['sos_home_norm'] < 0.97][['league','home_team','sos_home_norm','sos_home_class','z_gols_home','z_gols_home_adj']]
display(facil)

print()
print('=== Times com tabela difícil (SOS_norm > 1.03) — números mais legítimos ===')
dificil = ra[ra['sos_home_norm'] > 1.03][['league','home_team','sos_home_norm','sos_home_class','z_gols_home','z_gols_home_adj']]
display(dificil)

=== Top 10 — Maior Z_gols_home ajustado (mais preocupantes para Over) ===


,league,home_team,away_team,gols_marc_home,z_gols_home,sos_home_norm,z_gols_home_adj,sos_home_class
0,Alemanha_1 Bundesliga,Bayern München,Union Berlin,3.0,2.1851,1.0122,2.1588,🟡 Levemente difícil
4,Alemanha_1 Bundesliga,Borussia Dortmund,Hamburger SV,2.6,1.5890,0.9936,1.5992,⚪ Normal
113,Portugal_1 Primeira Liga,Sporting Braga,Porto,2.6,1.5890,1.0076,1.5770,⚪ Normal
35,Bulgaria_1 First League,Levski Sofia,Cherno More,2.5,1.4400,0.9929,1.4503,⚪ Normal
28,Belgica_1 Pro League,Club Brugge,KV Mechelen,2.4,1.2910,1.0007,1.2901,⚪ Normal
52,Espanha_1 La Liga,Real Madrid,Atlético Madrid,2.4,1.2910,1.0054,1.2841,⚪ Normal
83,Holanda_1 Eredivisie,NEC,Heerenveen,2.4,1.2910,1.0177,1.2685,🟡 Levemente difícil
16,Alemanha_3 3. Liga,Verl,Saarbrucken,2.3,1.1420,0.9885,1.1553,🟡 Levemente fácil
45,Escocia_1 Premiership,Rangers,Aberdeen,2.2,0.9930,0.9881,1.0050,🟡 Levemente fácil
116,Portugal_1 Primeira Liga,Benfica,Vitória Guimarães,2.2,0.9930,0.9936,0.9994,⚪ Normal



=== Times com tabela fácil (SOS_norm < 0.97) — números inflados ===


,league,home_team,sos_home_norm,sos_home_class,z_gols_home,z_gols_home_adj



=== Times com tabela difícil (SOS_norm > 1.03) — números mais legítimos ===


,league,home_team,sos_home_norm,sos_home_class,z_gols_home,z_gols_home_adj
75,Grecia_1 Super League,Levadiakos,1.0332,🔴 Tabela difícil,-0.1991,-0.1927


In [56]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 7 — Exportar CSV atualizado
# ═══════════════════════════════════════════════════════════

# Remover colunas auxiliares de classificação (só para análise interna)
ra.drop(columns=['sos_home_class','sos_away_class'], inplace=True, errors='ignore')

ra.to_csv(PATH_OUTPUT, index=False)

novas_colunas = [
    'sos_home_raw','sos_home_norm','sos_home_n',
    'sos_away_raw','sos_away_norm','sos_away_n',
    'z_gols_home','z_gols_home_adj',
    'z_gols_away','z_gols_away_adj',
    'z_xg_home','z_xg_home_adj',
    'z_xg_away','z_xg_away_adj',
    'z_vit_home','z_vit_home_adj',
    'z_vit_away','z_vit_away_adj',
]

print(f'✅ Exportado: {PATH_OUTPUT}')
print(f'   Shape final: {ra.shape}')
print()
print('Novas colunas adicionadas ao CSV:')
for c in novas_colunas:
    presente = '✓' if c in ra.columns else '✗'
    print(f'  {presente} {c}')

print()
print('📌 Próximo passo: envie o rodada_atual_v3.csv atualizado')
print('   para o Claude gerar o index.html com SOS embutido.')
display(ra.tail(3))

✅ Exportado: rodada_atual_sos_v3.csv
   Shape final: (154, 80)

Novas colunas adicionadas ao CSV:
  ✓ sos_home_raw
  ✓ sos_home_norm
  ✓ sos_home_n
  ✓ sos_away_raw
  ✓ sos_away_norm
  ✓ sos_away_n
  ✓ z_gols_home
  ✓ z_gols_home_adj
  ✓ z_gols_away
  ✓ z_gols_away_adj
  ✓ z_xg_home
  ✓ z_xg_home_adj
  ✓ z_xg_away
  ✓ z_xg_away_adj
  ✓ z_vit_home
  ✓ z_vit_home_adj
  ✓ z_vit_away
  ✓ z_vit_away_adj

📌 Próximo passo: envie o rodada_atual_v3.csv atualizado
   para o Claude gerar o index.html com SOS embutido.


,fixture_id,league,season_id,date_unix,home_id,away_id,home_team,away_team,status,odd_home_d2,...,z_gols_away,z_gols_away_adj,z_xg_home,z_xg_home_adj,z_xg_away,z_xg_away_adj,z_vit_home,z_vit_home_adj,z_vit_away,z_vit_away_adj
151,8432122,Colombia_1 Liga BetPlay,16614,1774143000,2142,2149,Atlético Nacional,La Equidad,incomplete,0.00,...,0.4048,0.4057,-0.1061,-0.1059,0.0191,0.0191,0.8135,0.8121,0.0387,0.0388
152,8432123,Colombia_1 Liga BetPlay,16614,1774308000,2150,2147,Jaguares de Córdoba,Rionegro Águilas,incomplete,2.61,...,-0.2860,-0.2878,-1.6196,-1.6414,-0.2091,-0.2104,-1.2223,-1.2388,0.0387,0.0389
153,8432124,Colombia_1 Liga BetPlay,16614,1774135200,2135,2136,Once Caldas,Millonarios,incomplete,2.14,...,0.7503,0.7514,-0.2144,-0.2123,0.8735,0.8748,-0.0008,-0.0008,0.4991,0.4998
